In [ ]:
from pathlib import Path

import prism

from imagematerials.eol import eol_preprocess
from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    EndOfLife,
    GenericMaterials,
    GenericStocks,
    Maintenance,
    MaterialIntensities,
)
from imagematerials.preprocessing import get_preprocessing_data

In [ ]:
scenario_list = {"SSP2_M_CP":("SSP2_M_CP", None)}

In [ ]:
scenario_base_path = Path("../data/raw") / 'circular_economy_scenarios'

time_start = 1721
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1721, 2060, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path("..", "data", "raw", "image", climate_scen)
    circular_economy_scenario_dirs = None

    bld_sector = get_preprocessing_data("buildings", Path("..", "data", "raw"), 
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dirs) 


    factory = ModelFactory(
    [bld_sector], complete_timeline
    ).add(GenericStocks, ["buildings"]
    ).add(MaterialIntensities, "buildings",
)
    model = factory.finish()

    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model.simulate(simulation_timeline)

    print(f"Finished {scen_id}")

In [ ]:
import matplotlib.pyplot as plt


bld_sector.prep_data.get("stocks").sel(Type = ['Appartment - Rural', 'Appartment - Urban', 'Detached - Rural',
       'Detached - Urban', 'High-rise - Rural', 'High-rise - Urban',
       'Semi-detached - Rural', 'Semi-detached - Urban']).sum(["Region", "Type"]).plot(label = 'residential')

bld_sector.prep_data.get("stocks").sel(Type = ['Office',
       'Retail+', 'Hotels+', 'Govt+']).sum(["Region", "Type"]).plot(label = 'gvt')
plt.legend()

In [ ]:
model.buildings.get("inflow").to_array().sel(Type = ['Appartment - Rural', 'Appartment - Urban', 'Detached - Rural',
       'Detached - Urban', 'High-rise - Rural', 'High-rise - Urban',
       'Semi-detached - Rural', 'Semi-detached - Urban']).sum(["Region", "Type"]).plot(label = 'residential')

model.buildings.get("inflow").to_array().sel(Type = ['Office',
       'Retail+', 'Hotels+', 'Govt+']).sum(["Region", "Type"]).plot(label = 'gvt')

plt.legend()

In [ ]:
model.buildings.get("inflow_materials").to_array().sel(material = 'steel', Type = ['Appartment - Rural', 'Appartment - Urban', 'Detached - Rural',
       'Detached - Urban', 'High-rise - Rural', 'High-rise - Urban',
       'Semi-detached - Rural', 'Semi-detached - Urban']).sum(["Region", "Type"]).plot(label = 'residential')

model.buildings.get("inflow_materials").to_array().sel(material = 'steel', Type = ['Office',
       'Retail+', 'Hotels+', 'Govt+']).sum(["Region", "Type"]).plot(label = 'gvt')

plt.legend()

In [ ]:
model.buildings.get("inflow_materials").to_array()

cement_concrete = model.buildings.get("inflow_materials").to_array().sel(material = 'concrete')*0.15
cement = model.buildings.get("inflow_materials").to_array().sel(material = 'cement')

(cement + cement_concrete).sum(["Type"]).sel(Region = "CHN").plot()



['Appartment - Rural', 'Appartment - Urban', 'Detached - Rural',
       'Detached - Urban', 'High-rise - Rural', 'High-rise - Urban',
       'Semi-detached - Rural', 'Semi-detached - Urban', 'Office', 'Retail+',
       'Hotels+', 'Govt+']

In [ ]:
# plot concrete use in China
model.buildings.get("inflow_materials").to_array().sel(material = 'cement').sel(Type = ['Appartment - Rural',
 'Appartment - Urban',
 'Detached - Rural',
 'Detached - Urban',
 'High-rise - Rural',
 'High-rise - Urban',
 'Semi-detached - Rural',
 'Semi-detached - Urban']).loc[2000:].sum(["Type"]).sel(Region = "CHN").plot(label = 'cement residential')

# plot concrete use in China
concrete_res = model.buildings.get("inflow_materials").to_array().sel(material = 'concrete').sel(Type = ['Appartment - Rural',
 'Appartment - Urban',
 'Detached - Rural',
 'Detached - Urban',
 'High-rise - Rural',
 'High-rise - Urban',
 'Semi-detached - Rural',
 'Semi-detached - Urban']).sum(["Type"]).loc[2000:].sel(Region = "CHN")
concrete_res = concrete_res*0.10
concrete_res.plot(label = 'cement (concrete) residential')

# plot concrete use in China
concrete_comm = model.buildings.get("inflow_materials").to_array().sel(material = 'concrete').sel(Type = ['Appartment - Rural',
 'Office', 'Retail+',
       'Hotels+', 'Govt+']).sum(["Type"]).sel(Region = "CHN").loc[2000:]
concrete_comm = concrete_comm*0.10
concrete_comm.plot(label = 'cement (concrete) commercial')

# plot concrete use in China
model.buildings.get("inflow_materials").to_array().sel(material = 'cement').sel(Type = ['Appartment - Rural',
 'Office', 'Retail+',
       'Hotels+', 'Govt+']).sum(["Type"]).sel(Region = "CHN").loc[2000:].plot(label = 'cement commercial')

plt.legend()